# HDB Resale Flat Prices — ETL Pipeline
**Senior Data Engineer Technical Test — Part 1: Developing Data Pipelines**

This notebook implements an end-to-end ETL pipeline for HDB resale flat transaction data (Jan 2012 – Dec 2016), covering:

1. **Extraction** — programmatic discovery & download of all relevant source files from data.gov.sg
2. **Data Profiling** — statistical profile of every column in the combined master dataset
3. **Validation** — rules for `month` (date), `town`, `flat_type`, `flat_model`, `storey_range`, derived from the dataset's own statistical properties
4. **Remaining lease recomputation** — assuming a 99-year lease
5. **Composite-key deduplication** — keep higher price on exact duplicates
6. **Anomalous price detection** — documented heuristic, flagged (not removed)
7. **Resale Identifier construction** — per the specified encoding rule
8. **Identifier-level deduplication** — keep higher price on identifier collisions
9. **Irreversible hashing** of the identifier, preserving row-level uniqueness
10. **Output** — Raw / Cleaned / Transformed / Failed / Hashed datasets

### Engineering approach
Pipeline logic lives in importable, unit-testable modules under `src/` (`extract.py`, `profile.py`, `validate.py`, `transform.py`, `hash_utils.py`, `pipeline.py`). This notebook **orchestrates and documents** those modules step-by-step so each requirement is traceable to its output — rather than burying all logic in notebook cells, which is harder to test, review, and reuse in a production scheduler (e.g. Airflow / Step Functions, see Part 2).

### Reproducibility note
This notebook was authored and run in a **sandboxed environment with no outbound internet access**. `extract.download_datasets()` is fully implemented against data.gov.sg's public API and **will run end-to-end in a networked environment** (your machine / CI / Airflow worker) to fetch every relevant vintage file automatically. In this sandbox it gracefully no-ops (logged as a warning below) and the pipeline falls back to the single sample file provided (`Resale Flat Prices Based on Registration Date, Mar 2012-Dec 2014`, 52,203 rows). All downstream logic is written generically over N input files, not hardcoded to this one — dropping the remaining vintage files (2012 Jan–Feb, 2015–2016) into `data/raw/` and re-running reproduces the full Jan 2012–Dec 2016 master dataset with no code changes.

In [1]:
import sys, logging
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "src"))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

RAW_DIR = Path.cwd() / "data" / "raw"
OUTPUT_DIR = Path.cwd() / "output"
for sub in ("raw", "cleaned", "transformed", "failed", "hashed"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Extraction

`extract.download_datasets()` calls data.gov.sg's public Collections API to discover every dataset in the "Resale Flat Prices" collection, downloads the ones overlapping our Jan 2012–Dec 2016 window, and saves them as-is into `data/raw/` — no manual interface interaction. `extract.load_raw_files()` then reads **every** CSV in `data/raw/` and unions their schemas (outer join on columns), so the combined master dataset contains every attribute found in every file, even if a given file doesn't have that column (filled as null for those rows). Each row is tagged with `_source_file` and a stable `_row_id` for lineage.

In [2]:
from extract import download_datasets, load_raw_files

download_datasets(RAW_DIR)          # no-ops with a warning if network is unavailable (see note above)
raw_df = load_raw_files(RAW_DIR)    # unions schemas across every CSV present in data/raw/

raw_df.to_csv(OUTPUT_DIR / "raw" / "master_raw.csv", index=False)
print(f"Master RAW dataset: {len(raw_df):,} rows x {raw_df.shape[1]} columns")
raw_df.head()

Master RAW dataset: 52,203 rows x 12 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,_source_file,_row_id
0,2012-03,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45,Improved,1986,250000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,1
1,2012-03,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44,Improved,1980,265000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,2
2,2012-03,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68,New Generation,1980,315000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,3
3,2012-03,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67,New Generation,1984,320000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,4
4,2012-03,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67,New Generation,1980,321000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,5


## 2. Data Profiling

`profile.profile_dataset()` computes, per column: dtype, null count/%, distinct count/%, and (for categorical columns) the value-frequency distribution including a statistically-derived "rare value" long tail (< 0.05% frequency), and (for numeric columns) min/max/mean/std/quartiles.

This profile is the **statistical basis** for the validation rules in the next section — satisfying the requirement that validation be "based on the statistical properties of this master dataset" rather than an externally hardcoded lookup list.

In [3]:
from profile import profile_dataset, print_profile

report = profile_dataset(raw_df)
print_profile(report)


=== month ===
  dtype: str
  non_null: 52203
  nulls: 0
  null_pct: 0.0
  distinct: 34
  distinct_pct: 0.065

=== town ===
  dtype: str
  non_null: 52203
  nulls: 0
  null_pct: 0.0
  distinct: 26
  distinct_pct: 0.05
  top_values:
      'WOODLANDS': 4502
      'JURONG WEST': 4391
      'TAMPINES': 3821
      'YISHUN': 3588
      'BEDOK': 3436
      'SENGKANG': 3094
      'HOUGANG': 2667
      'ANG MO KIO': 2553
      'BUKIT BATOK': 2142
      'BUKIT MERAH': 2133
  rare_values: none

=== flat_type ===
  dtype: str
  non_null: 52203
  nulls: 0
  null_pct: 0.0
  distinct: 7
  distinct_pct: 0.013
  top_values:
      '4 ROOM': 20150
      '3 ROOM': 15372
      '5 ROOM': 11845
      'EXECUTIVE': 4194
      '2 ROOM': 581
      '1 ROOM': 40
      'MULTI-GENERATION': 21
  rare_values (1): ['MULTI-GENERATION']

=== block ===
  dtype: str
  non_null: 52203
  nulls: 0
  null_pct: 0.0
  distinct: 2047
  distinct_pct: 3.921

=== street_name ===
  dtype: str
  non_null: 52203
  nulls: 0
  null_pct: 

## 3. Validation Rules

Implemented in `validate.validate_dataset()`. All rules are **derived from the profile above**:

| Field | Rule |
|---|---|
| `month` | must match `YYYY-MM` format |
| `town` | flagged if null, or if it falls in the statistically-derived rare/long-tail set (< 0.05% frequency in the master dataset) |
| `flat_type` | same rare-value logic as `town` |
| `flat_model` | same rare-value logic as `town` |
| `storey_range` | must match `NN TO NN` format, and the lower bound must not exceed the upper bound |

**Additional rules** (Data Quality Requirement #7): `floor_area_sqm` must be numeric and within a sane physical range [20, 500] sqm; `resale_price` must be numeric and ≥ $1,000; `lease_commence_date` must be a plausible year; `block` / `street_name` must not be blank.

**Caveat on the 0.05% rarity threshold:** this is a tunable heuristic, not a hard business rule. In this sample, `MULTI-GENERATION` flat types (21 of 52,203 rows ≈ 0.040%) fall just under the threshold and get flagged — even though it's a legitimate HDB flat_type, simply uncommon in this window. In a production setting we'd recommend a human review pass over `report[col]['rare_values']` before finalising an exclusion list (or seeding the valid-set from a longer historical window so genuine-but-uncommon categories aren't misclassified). This is intentionally surfaced, not silently "fixed", to demonstrate the mechanism as required.

In [4]:
from validate import validate_dataset

validated_passed_df, failed_validation_df = validate_dataset(raw_df, report)
print(f"Passed validation: {len(validated_passed_df):,}   Failed validation: {len(failed_validation_df):,}")
failed_validation_df["_fail_reason"].value_counts()

Passed validation: 52,171   Failed validation: 32


_fail_reason
flat_type is null or a statistical rarity/outlier (<0.05% frequency); flat_model is null or a statistical rarity/outlier (<0.05% frequency)    21
flat_model is null or a statistical rarity/outlier (<0.05% frequency)                                                                          11
Name: count, dtype: int64

## 4. Composite-Key Deduplication

Per the assignment: the composite key is *all columns except `resale_price`*. Where two rows share an identical composite key, we keep the row with the **higher** price and discard the rest into the failed dataset (`transform.dedup_composite_key()`).

In [5]:
from transform import dedup_composite_key

deduped_df, composite_dup_df = dedup_composite_key(validated_passed_df)
print(f"Kept: {len(deduped_df):,}   Discarded as composite-key duplicates: {len(composite_dup_df):,}")
composite_dup_df.head(3)

Kept: 51,086   Discarded as composite-key duplicates: 1,085


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,_source_file,_row_id,_fail_reason
44,2012-03,ANG MO KIO,3 ROOM,173,ANG MO KIO AVE 4,01 TO 05,83,New Generation,1982,372000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,45,duplicate composite key (all columns except resale_price identical); lower price discarded
98,2012-03,ANG MO KIO,5 ROOM,648,ANG MO KIO AVE 5,01 TO 05,121,Improved,1980,538000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,99,duplicate composite key (all columns except resale_price identical); lower price discarded
101,2012-03,ANG MO KIO,5 ROOM,716,ANG MO KIO AVE 6,06 TO 10,119,Improved,1980,590000,Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv,102,duplicate composite key (all columns except resale_price identical); lower price discarded


## 5. Remaining Lease Recomputation

**Assumption:** the source data only gives the lease-commencement *year*; we assume the lease commences 1 Jan of that year. Remaining lease = (lease start + 99 years) − today, **floored** to whole months (partial months dropped), matching HDB's own "X years Y months" convention. `transform.compute_remaining_lease()`.

In [6]:
from transform import compute_remaining_lease

cleaned_df = compute_remaining_lease(deduped_df)
cleaned_df[["lease_commence_date", "remaining_lease_years", "remaining_lease_months", "remaining_lease_display"]].head()

,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_display
0,1986,58,4,58 years 04 months
1,1980,52,4,52 years 04 months
2,1980,52,4,52 years 04 months
3,1984,56,4,56 years 04 months
4,1980,52,4,52 years 04 months


## 6. Anomalous Resale Price Heuristic

**Heuristic (documented per Data Quality Requirement #6):** we compute **price per square metre** — a fairer normalising metric than raw price, since unit size varies substantially — grouped by `(town, flat_type)`. Within each group, a transaction is flagged as a potential anomaly if its price-per-sqm falls outside **Q1 − 3×IQR, Q3 + 3×IQR** (a wider-than-conventional 3× multiplier, vs. the usual 1.5×, because resale prices are legitimately right-skewed and we want to surface only genuine outliers for analyst review, not the normal tail of the distribution).

This is a **soft flag**, not a hard rejection: anomalous rows stay in the Cleaned dataset with `is_price_anomaly=True` and a `price_anomaly_reason`, for the Data Science Team to inspect — a suspiciously high/low price could be a genuine outlier transaction (e.g. resale with heavy renovation, distressed sale) rather than bad data, so we don't want to silently discard it.

In [7]:
from transform import flag_price_anomalies

cleaned_df = flag_price_anomalies(cleaned_df)
cleaned_df.to_csv(OUTPUT_DIR / "cleaned" / "master_cleaned.csv", index=False)

print(f"CLEANED dataset written: {len(cleaned_df):,} rows, {cleaned_df['is_price_anomaly'].sum()} flagged as price anomalies (kept, not removed)")
cleaned_df[cleaned_df["is_price_anomaly"]][["month","town","flat_type","floor_area_sqm","resale_price","price_per_sqm"]].head()

CLEANED dataset written: 51,086 rows, 161 flagged as price anomalies (kept, not removed)


,month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm
395,2012-03,BUKIT BATOK,4 ROOM,90,602000,6688.888889
1262,2012-03,KALLANG/WHAMPOA,3 ROOM,80,688000,8600.000000
1263,2012-03,KALLANG/WHAMPOA,3 ROOM,83,698000,8409.638554
3500,2012-04,KALLANG/WHAMPOA,3 ROOM,62,565000,9112.903226
5305,2012-05,GEYLANG,3 ROOM,60,454000,7566.666667


## 7. Data Transformation — Resale Identifier

Per the specification, `Resale Identifier = S + BBB + PP + MM + T`:

- **`S`** — literal first character
- **`BBB`** — first 3 digits of `block`, after stripping non-digit characters, zero-padded on the left if fewer than 3 digits (e.g. block `"19"` → `"019"`)
- **`PP`** — first 2 digits of the **average resale price**, grouped by `(year-month, town, flat_type)`, for this row's group
- **`MM`** — the month portion of this row's `month` field (`YYYY-MM` → `MM`)
- **`T`** — first character of `town`

Implemented in `transform.build_resale_identifier()`.

In [8]:
from transform import build_resale_identifier

with_id_df = build_resale_identifier(cleaned_df)
with_id_df[["month","town","flat_type","block","resale_price","resale_identifier"]].head(8)

,month,town,flat_type,block,resale_price,resale_identifier
0,2012-03,ANG MO KIO,2 ROOM,172,250000,S1722503A
1,2012-03,ANG MO KIO,2 ROOM,510,265000,S5102503A
2,2012-03,ANG MO KIO,3 ROOM,610,315000,S6103603A
3,2012-03,ANG MO KIO,3 ROOM,474,320000,S4743603A
4,2012-03,ANG MO KIO,3 ROOM,604,321000,S6043603A
5,2012-03,ANG MO KIO,3 ROOM,154,321000,S1543603A
6,2012-03,ANG MO KIO,3 ROOM,110,323000,S1103603A
7,2012-03,ANG MO KIO,3 ROOM,445,325000,S4453603A


## 8. Post-Identifier Deduplication

The Resale Identifier is a **derived, lossy** code — it's possible (by construction) for two genuinely different transactions to collide onto the same identifier string even after composite-key dedup. Per the spec: where duplicates occur, keep the higher price and discard the rest (`transform.dedup_by_identifier()`).

In [9]:
from transform import dedup_by_identifier

transformed_df, identifier_dup_df = dedup_by_identifier(with_id_df)
transformed_df.to_csv(OUTPUT_DIR / "transformed" / "master_transformed.csv", index=False)

print(f"TRANSFORMED dataset written: {len(transformed_df):,} rows")
print(f"Discarded for duplicate resale_identifier: {len(identifier_dup_df):,}")
assert transformed_df["resale_identifier"].is_unique

TRANSFORMED dataset written: 44,402 rows
Discarded for duplicate resale_identifier: 6,684


## 9. Hashing the Resale Identifier

**Algorithm: SHA-256** (`hash_utils.py`), computed as `SHA256(resale_identifier + "|" + _row_id)`.

- **Irreversible:** SHA-256 is a cryptographic one-way hash — pre-image resistance makes it computationally infeasible to recover the original identifier from the digest.
- **Uniqueness-preserving:** at 256 bits of output, accidental collisions are astronomically unlikely (birthday-bound ≈ 2^128) — in practice, distinct inputs never collide.
- **Deterministic:** identical input always produces identical output, which matters for reproducibility and downstream joins.
- **Why mix in `_row_id`:** the identifier itself is lossy, so two *different* surviving rows could in principle still share an identifier string in edge cases. Hashing `identifier + row_id` guarantees the hash is unique **per row**, not merely per distinct identifier value. If instead the goal were "identical identifiers should hash identically" (e.g. for grouping), `hash_utils.hash_identifier_only()` is provided as an alternative.

In [10]:
from hash_utils import add_hashed_identifier

hashed_df = add_hashed_identifier(transformed_df)
hashed_df.to_csv(OUTPUT_DIR / "hashed" / "master_hashed.csv", index=False)

print(f"HASHED dataset written: {len(hashed_df):,} rows")
print(f"Unique hashes: {hashed_df['resale_identifier_hash'].nunique():,} / {len(hashed_df):,}")
hashed_df[["resale_identifier", "resale_identifier_hash"]].head(3)

HASHED dataset written: 44,402 rows
Unique hashes: 44,402 / 44,402


,resale_identifier,resale_identifier_hash
0,S1722503A,e1a40ad1366d04ff7021389776e74b0a561c36502df3ca61cc4ee7490e75618c
1,S5102503A,d268acdbe20fe8777a8bf477ea75783e175a22d15886afec3ed1331634ad17d4
2,S6103603A,449c935a78ba1fcc96a9cb584bd9b06826631f8e8a25da65620c0800c4e5cb97


## 10. Failed Dataset & Output Summary

All discarded/rejected rows across every stage (validation failures, composite-key duplicates, identifier duplicates) are consolidated into a single **Failed** dataset with a `_fail_reason` column, for full auditability.

In [11]:
all_failed = pd.concat(
    [df for df in [failed_validation_df, composite_dup_df, identifier_dup_df] if len(df)],
    axis=0, ignore_index=True, sort=False,
)
all_failed.to_csv(OUTPUT_DIR / "failed" / "master_failed.csv", index=False)

print("=== PIPELINE OUTPUT SUMMARY ===")
print(f"Raw:         {len(raw_df):,} rows  -> output/raw/master_raw.csv")
print(f"Cleaned:     {len(cleaned_df):,} rows  -> output/cleaned/master_cleaned.csv")
print(f"Transformed: {len(transformed_df):,} rows  -> output/transformed/master_transformed.csv")
print(f"Hashed:      {len(hashed_df):,} rows  -> output/hashed/master_hashed.csv")
print(f"Failed:      {len(all_failed):,} rows  -> output/failed/master_failed.csv")
all_failed["_fail_reason"].value_counts()

=== PIPELINE OUTPUT SUMMARY ===
Raw:         52,203 rows  -> output/raw/master_raw.csv
Cleaned:     51,086 rows  -> output/cleaned/master_cleaned.csv
Transformed: 44,402 rows  -> output/transformed/master_transformed.csv
Hashed:      44,402 rows  -> output/hashed/master_hashed.csv
Failed:      7,801 rows  -> output/failed/master_failed.csv


_fail_reason
duplicate resale_identifier; lower price discarded                                                                                             6684
duplicate composite key (all columns except resale_price identical); lower price discarded                                                     1085
flat_type is null or a statistical rarity/outlier (<0.05% frequency); flat_model is null or a statistical rarity/outlier (<0.05% frequency)      21
flat_model is null or a statistical rarity/outlier (<0.05% frequency)                                                                            11
Name: count, dtype: int64

## Appendix — Assumptions & Design Decisions

1. **Date window generalisation:** pipeline logic (extraction → union → profiling → validation → transformation) is written generically over however many raw CSVs are present in `data/raw/`, so it directly generalises from this sample (Mar 2012–Dec 2014) to the full Jan 2012–Dec 2016 requirement once the remaining vintage files are downloaded (automatic, given network access) or dropped in locally.
2. **Schema drift across vintages:** earlier/later data.gov.sg files may have different columns (e.g. `remaining_lease` provided directly in 2017+ data). The union-of-columns approach in `extract.load_raw_files()` means no attribute is silently dropped; a later file that already provides `remaining_lease` would appear as an extra, separate column alongside our recomputed one for reconciliation.
3. **Rarity threshold (0.05%)** for categorical validation is a tunable constant (`profile.RARE_VALUE_FREQ_THRESHOLD`), not a hardcoded enum — satisfying "based on statistical properties" but flagged above as needing analyst sign-off before being treated as ground truth.
4. **Lease start date:** assumed 1 Jan of `lease_commence_date`'s year, since only the year is provided.
5. **Anomaly detection is a soft flag, not a filter** — anomalous rows remain in the Cleaned/Transformed/Hashed outputs, distinguishable via `is_price_anomaly`, since a statistical outlier is not necessarily bad data.
6. **Hash design:** `identifier + row_id` guarantees row-level hash uniqueness even though the identifier itself is a lossy, non-unique code by construction (see Section 9).